# **EDA BRONZE ANALYSIS**

# **RESUMEN EJECUTIVO**

- **Total de Registros analizados:** 6567
- **Periodo Análisis:** 01-01-2025 ==> 30-06-2026
- **Cantidad Productos Únicos:** 31
- **Cantidad Nombres Mercados Únicos:** 14
- **Cantidad Unidad de medidas Únicos:** 13

## **Problemas de Calidad**

- **Valores nulos:** Se encontraron valores nulos en nombre_producto y nombre_mercado
- **Posibles Datos Duplicados:** Registros de productos repetidos
- **Valores fuera de rango:** Fechas fuera del rango estimado


# **Executive Review**

- **Analized Total Records:** 6567
- **Analysis Time Interval:** 01-01-2025 ==> 06-30-2026
- **Unique Products List:** 31
- **Unique Market Names:** 14
- **Unique Unit Measure:** 13

## **Quality Problems Found**

- **Null Values:** Several record have null values in the fields(product names) and market_name
- **Duplicated records:** Duplicated records have been found
- **Out of range values:** Dates out of range 

In [0]:
%sql
------------------------------------
-- Step 1: Table Structure
------------------------------------

DESCRIBE TABLE EXTENDED products.bronze.product_list_markets_bronze

In [0]:
%sql
select count(*) from products.bronze.product_list_markets_bronze

In [0]:
%sql
------------------------------------
-- Step 2: Count Null Values per Column
------------------------------------


select
  count(*) - count(nombre_mercado) as nombre_mercado_null,
  count(*) - count(product_list_id) as product_list_id_null,
  count(*) - count(unidad_medida) as unidad_medida_null,
  count(*) - count(inicio_fecha_precios) as inicio_fecha_precios_null,
  count(*) - count(fin_fecha_precios) as fin_fecha_precios_null,
  count(*) - count(precio_inicio) as precio_inicio_null,
  count(*) - count(precio_fin) as precio_fin_null,
  count(*) - count(error) as error_null
from
  products.bronze.product_list_markets_bronze

In [0]:
%sql
-- Null Values Percentage

with total_values as(
    select count(*) as total from products.bronze.product_list_markets_bronze
)

select
  round((tt.total - count(nombre_mercado)) * 100.0 / tt.total, 2) as nombre_mercado_null_pct,
  round((tt.total - count(product_list_id)) * 100.0 / tt.total, 2) as product_list_id_null_pct,
  round((tt.total - count(unidad_medida)) * 100.0 / tt.total, 2) as unidad_medida_null_pct,
  round((tt.total - count(inicio_fecha_precios)) * 100.0 / tt.total, 2) as inicio_fecha_precios_null_pct,
  round((tt.total - count(fin_fecha_precios)) * 100.0 / tt.total, 2) as fin_fecha_precios_null_pct,
  round((tt.total - count(precio_inicio)) * 100.0 / tt.total, 2) as precio_inicio_null_pct,
  round((tt.total - count(precio_fin)) * 100.0 / tt.total, 2) as precio_fin_null_pct
from
  products.bronze.product_list_markets_bronze
cross join total_values tt
group by tt.total

In [0]:
%sql
------------------------------------
-- Step 3: Percentages Market Distribution
------------------------------------


select distinct
  nombre_mercado,
  count(*) as total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentage
from
  products.bronze.product_list_markets_bronze
group by nombre_mercado
order by total desc

In [0]:
%sql
-- Percentages Product Distribution

select distinct
  nombre_producto,
  count(*) as total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentage
from
  products.bronze.product_list_markets_bronze
group by nombre_producto
order by total desc

In [0]:
%sql
-- Percentages Measurement Distribution

select distinct
  unidad_medida,
  count(*) as total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentage
from
  products.bronze.product_list_markets_bronze
group by unidad_medida
order by total desc

In [0]:
%sql

create or replace temporary view products_bronze_transform_analysis as
select
  nombre_mercado,
  nombre_producto,
  unidad_medida,
  error,
  case
    when
      precio_inicio rlike '^-?[0-9]+\\.?[0-9]*$'
      and precio_inicio::double between 0 and 2147483648
    then
      precio_inicio::double
    else null
  end as precio_inicio_clean,
  case
    when
      precio_fin rlike '^-?[0-9]+\\.?[0-9]*$'
      and precio_fin::double between 0 and 2147483648
    then
      precio_fin::double
    else null
  end as precio_fin_clean,
  case
    when inicio_fecha_precios RLIKE '^[^a-zA-Z]+$' then to_date(inicio_fecha_precios, 'dd-MM-yyyy')
    else null
  end as inicio_fecha_precios_clean,
  case
    when fin_fecha_precios RLIKE '^[^a-zA-Z]+$' then to_date(fin_fecha_precios, 'dd-MM-yyyy')
    else null
  end as fin_fecha_precios_clean
from
  products.bronze.product_list_markets_bronze

In [0]:
%sql
------------------------------------
-- Step 4: Ranges
------------------------------------

select
  max(precio_inicio_clean) as max_precio_inicio_clean,
  min(precio_inicio_clean) as min_precio_inicio_clean,
  round(avg(precio_inicio_clean), 2) as avg_precio_inicio_clean,
  max(precio_fin_clean) as max_precio_fin_clean,
  min(precio_fin_clean) as min_precio_fin_clean,
  round(avg(precio_fin_clean), 2) as avg_precio_fin_clean,
  max(inicio_fecha_precios_clean) as max_inicio_fecha_precios_clean,
  min(inicio_fecha_precios_clean) as min_inicio_fecha_precios_clean,
  max(fin_fecha_precios_clean) as max_fin_fecha_precios_clean,
  min(fin_fecha_precios_clean) as min_fin_fecha_precios_clean
from
  products_bronze_transform_analysis

In [0]:
%sql
------------------------------------
-- Step 4b: Date Outlier Filter (1st-99th Percentile)
------------------------------------

with dates_range as(
    select
    PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY unix_date(inicio_fecha_precios_clean)) AS fip_001,
    PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY unix_date(inicio_fecha_precios_clean)) AS fip_099,
    PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY unix_date(fin_fecha_precios_clean)) AS ffp_001,
    PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY unix_date(fin_fecha_precios_clean)) AS ffp_099
    from 
    products_bronze_transform_analysis
)

select
    p.nombre_mercado,
    p.nombre_producto,
    p.unidad_medida,
    p.precio_inicio_clean,
    p.precio_fin_clean,
    p.inicio_fecha_precios_clean,
    p.fin_fecha_precios_clean
    from 
    products_bronze_transform_analysis p

cross join dates_range d
where p.inicio_fecha_precios_clean between date_add('1970-01-01', cast(d.fip_001 as int)) and date_add('1970-01-01', cast(d.fip_099 as int))
and p.fin_fecha_precios_clean between date_add('1970-01-01', cast(d.ffp_001 as int)) and date_add('1970-01-01', cast(d.ffp_099 as int))

In [0]:
%sql
------------------------------------
-- Step 5: Counting Duplicates
------------------------------------


WITH duplicados AS (
  SELECT
    nombre_mercado,
    nombre_producto,
    unidad_medida,
    precio_inicio_clean,
    precio_fin_clean,
    inicio_fecha_precios_clean,
    fin_fecha_precios_clean,
    COUNT(*) as veces
  FROM
    products_bronze_transform_analysis
  GROUP BY
    nombre_mercado,nombre_producto,unidad_medida,precio_inicio_clean,precio_fin_clean,inicio_fecha_precios_clean,fin_fecha_precios_clean
  HAVING
    COUNT(*) > 1
)

SELECT
  *
FROM
  duplicados
ORDER BY
  veces DESC
LIMIT 10;

In [0]:
%sql
------------------------------------
-- Step 6: Time Distribution
------------------------------------

SELECT 
    DATE_FORMAT(DATE_TRUNC('month', inicio_fecha_precios_clean),'yyyy-MM') as mes,
    COUNT(*) as product_quantity,
    ROUND(AVG(precio_inicio_clean), 2) as precio_promedio_inicial,
    ROUND(AVG(precio_fin_clean), 2) as precio_promedio_final
FROM products_bronze_transform_analysis
WHERE inicio_fecha_precios_clean IS NOT NULL
  AND precio_inicio_clean > 0
GROUP BY DATE_TRUNC('month', inicio_fecha_precios_clean),
DATE_TRUNC('year', inicio_fecha_precios_clean)
ORDER BY mes;
